In [ ]:
!git clone https://github.com/Mickrbl/TaxiNY.git
%cd TaxiNY

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime
import gc

YELLOW_DIR = '/kaggle/working/TaxiNY/data/taxi_gialli'
GREEN_DIR = '/kaggle/working/TaxiNY/data/taxi_verdi'
FHV_DIR = '/kaggle/working/TaxiNY/data/fhv'
HVFHV_DIR = '/kaggle/input/datasets/matteopetrelli/high-volume-fhv/High_Volume_FHV'

TARGET_YEAR = 2025

# Liste leggere in cui salveremo solo i "riassunti" di ogni file
demand_chunks = []
supply_chunks = []

def process_and_summarize(directory, taxi_label):
    files = glob.glob(os.path.join(directory, "*.parquet"))
    
    # Mappatura colonne
    if taxi_label == 'yellow':
        cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID']
    elif taxi_label == 'green':
        cols = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID']
    elif taxi_label == 'fhv':
        cols = ['pickup_datetime', 'dropOff_datetime', 'PUlocationID', 'DOlocationID']
    elif taxi_label == 'hvfhv':
        cols = ['pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID']
    
    for f in files:
        try:
            # 1. Carichiamo un SINGOLO file alla volta
            tmp = pd.read_parquet(f, columns=cols, engine='pyarrow')
            tmp.columns = ['pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID']
            
            # 2. Pulizia immediata
            tmp = tmp.dropna()
            tmp = tmp[(tmp['pickup_datetime'].dt.year == TARGET_YEAR) & 
                      (tmp['dropoff_datetime'].dt.year == TARGET_YEAR)]
            tmp = tmp[tmp['dropoff_datetime'] > tmp['pickup_datetime']]
            
            if len(tmp) == 0:
                continue
                
            # 3. Compressione RAM estrema
            tmp['PULocationID'] = tmp['PULocationID'].astype('uint16')
            tmp['DOLocationID'] = tmp['DOLocationID'].astype('uint16')
            tmp['taxi_type'] = taxi_label
            tmp['taxi_type'] = tmp['taxi_type'].astype('category')
            
            # ==========================================
            # MAP: CALCOLO DEI RIASSUNTI (Riduce 20M di righe a poche migliaia)
            # ==========================================
            
            # --- Domanda ---
            tmp['pickup_date'] = tmp['pickup_datetime'].dt.date
            tmp['pickup_hour'] = tmp['pickup_datetime'].dt.hour
            
            demand_summary = tmp.groupby(
                ['taxi_type', 'PULocationID', 'pickup_date', 'pickup_hour'], observed=True
            ).size().reset_index(name='predicted_demand')
            demand_chunks.append(demand_summary)
            
            # --- Offerta (con Shift di 1 ora) ---
            tmp['dropoff_datetime_shifted'] = tmp['dropoff_datetime'] + pd.Timedelta(hours=1)
            tmp['dropoff_date'] = tmp['dropoff_datetime_shifted'].dt.date
            tmp['dropoff_hour'] = tmp['dropoff_datetime_shifted'].dt.hour
            
            supply_summary = tmp.groupby(
                ['taxi_type', 'DOLocationID', 'dropoff_date', 'dropoff_hour'], observed=True
            ).size().reset_index(name='predicted_supply')
            supply_chunks.append(supply_summary)
            
            # 4. REDUCE: Distruggiamo il file grosso dalla RAM!
            del tmp, demand_summary, supply_summary
            gc.collect()
            
        except Exception as e:
            print(f"⚠️ Errore con {f}: {e}")

print("🚀 Inizio fase MAP-REDUCE: Elaborazione file per file in corso...")
process_and_summarize(YELLOW_DIR, 'yellow')
process_and_summarize(GREEN_DIR, 'green')
process_and_summarize(FHV_DIR, 'fhv')
process_and_summarize(HVFHV_DIR, 'hvfhv')

print("✅ Lettura file completata. Unione dei riassunti in corso...")

# 5. Uniamo tutti i piccoli blocchi riassuntivi
demand_df_raw = pd.concat(demand_chunks, ignore_index=True)
supply_df_raw = pd.concat(supply_chunks, ignore_index=True)

del demand_chunks, supply_chunks
gc.collect()

# 6. SECONDA AGGREGAZIONE (Uniamo i dati dello stesso giorno tagliati su file diversi)
demand_df = demand_df_raw.groupby(
    ['taxi_type', 'PULocationID', 'pickup_date', 'pickup_hour'], observed=True
)['predicted_demand'].sum().reset_index()

supply_df = supply_df_raw.groupby(
    ['taxi_type', 'DOLocationID', 'dropoff_date', 'dropoff_hour'], observed=True
)['predicted_supply'].sum().reset_index()

del demand_df_raw, supply_df_raw
gc.collect()

# ==========================================
# 7. FORMATTAZIONE FINALE (Date, Mesi, Weekend)
# ==========================================
giorni_ordinati = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Domanda
demand_df['pickup_date'] = pd.to_datetime(demand_df['pickup_date'])
demand_df['month'] = demand_df['pickup_date'].dt.month
demand_df['week_of_year'] = demand_df['pickup_date'].dt.isocalendar().week
demand_df['day_of_week'] = pd.Categorical(demand_df['pickup_date'].dt.day_name(), categories=giorni_ordinati, ordered=True)
demand_df['is_weekend'] = demand_df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)
demand_df = demand_df.rename(columns={'PULocationID': 'zone_id', 'pickup_date': 'date', 'pickup_hour': 'hour'})

# Offerta
supply_df['dropoff_date'] = pd.to_datetime(supply_df['dropoff_date'])
supply_df['month'] = supply_df['dropoff_date'].dt.month
supply_df['week_of_year'] = supply_df['dropoff_date'].dt.isocalendar().week
supply_df['day_of_week'] = pd.Categorical(supply_df['dropoff_date'].dt.day_name(), categories=giorni_ordinati, ordered=True)
supply_df['is_weekend'] = supply_df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)
supply_df = supply_df.rename(columns={'DOLocationID': 'zone_id', 'dropoff_date': 'date', 'dropoff_hour': 'hour'})

print(f"🎉 FASE 1 COMPLETATA: {len(demand_df):,} righe aggregate di Domanda | {len(supply_df):,} righe di Offerta.")

